# N094 · 覆盖前沿、跳跃与层次贪心

**目标：** 把可达位置集合压缩为最远边界。

**先修：** N013, N081。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 最远可达；当前层/下一层；不可达；贪心领先性。

**配套讲解来源：** [同名逐章意见](../../comment/094_greedy_reachability_layers.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

你站在数组的“学习目标”部分，每个格子上写着一个非负整数 `nums[i]`，表示"你站在这格时，最多向前跳这么多步"。两个经典问题：

1. **你能不能跳到最后一格？**（LeetCode 55，`can_jump`）
2. **至少跳几次能到最后一格？**（LeetCode 45，`min_jumps`；跳不到时本章约定返回 -1）

给你具体数字（就是 “运行示例”部分 的实例）：`nums=[2,3,1,1,4]`。**能不能到？能。最少几跳？2 跳**——先从位置 0 跳到位置 1（0 的跳长上界是 2，所以跳 1 步落在本层范围内合法），再从位置 1（跳长 3）直接跳到位置 4。为什么不是 1 跳？因为位置 0 最多跳到位置 2，一跳够不到终点；为什么不是必须 3 跳？因为上面那条两跳路线存在。再看一个到不了的例子：`nums=[3,2,1,0,4]`——不管你怎么跳，都会踩到位置 3，而位置 3 的跳长是 0，你就被钉死在那里，位置 4 永远够不着，所以 `can_jump` 返回 False、`min_jumps` 返回 -1。

这章的核心思想是：别把每个位置当成图上的点去做完整 BFS，而是发现"同一跳数能到的位置连成一段连续区间"，只要记住这段区间的右边界就够了。

## 2. 基础知识：先读懂这些词

- **跳跃游戏（jump game）**：从数组位置 0 出发，按每格给的步数上限向前跳，问可达性与最少跳数。它是"隐式图搜索"的典型题——图没有直接给你，位置是点、跳跃是边。
- **可达（reachable）**：存在一串合法跳跃从 0 走到某个位置。0 跳能到的只有位置 0 自己。
- **前沿 / 最远可达（frontier, far）**：一个数 `far`，表示"用目前的跳数，最远能摸到哪个位置"。它把"整段可达区间"压缩成了一个右边界，这就是 “学习目标”部分 说的"把可达位置集合压缩为最远边界"。
- **当前层 / 下一层（current layer / next layer）**：把"恰好 k 跳能到的所有位置"看成 BFS 的第 k 层。本章的关键观察是：每一层一定是一段连续的前缀区间，所以第 k 层可以用它的右边界 `end` 一个数表示；扫描时顺手算出的 `far` 就是"下一层的右边界"。
- **层次贪心（layered greedy）**：`min_jumps` 的做法——一层一层向外扩，扫完当前层的所有位置才把跳数加一。它完全对应 BFS 的层数，所以跳数天然最少。
- **贪心领先性**：`far=max(far, i+nums[i])` 这一步永远把前沿推到"目前已知最远"，绝不回退。它的合法性来自"跳长上限以内任何落点都合法"，所以前沿只增不减。
- **不可达（stuck）**：当扫描推进到位置 i 时 `i>far`，说明出现了一个前沿覆盖不到的空隙；数组从左往右扫，空隙后面的位置全都过不去，直接判死。另一种死法是下一层前沿和当前层右边界重合（`far==end`），前沿不再增长，后面永远到不了。

## 3. 思路推导：从小例子开始

**can_jump 的思路：**

1. **Step 1：** 维护一个前沿 `far`，初始为 0（0 跳最远只能到位置 0）。
2. **Step 2：** 从左往右扫每个位置 i。如果 `i>far`，说明位置 i 本身就到不了，而后面的更到不了，返回 False。
3. **Step 3：** 否则用位置 i 更新前沿 `far=max(far, i+nums[i])`；一旦 `far` 覆盖到最后一格就立刻返回 True。

**min_jumps 的思路（层次版）：**

1. **Step 1：** 维护三个数：`end`（当前层的右边界）、`far`（下一层的右边界）、`jumps`（已经用的跳数）。
2. **Step 2：** 从左往右扫位置 i（只扫到倒数第二格，因为跳到最后一格时循环就该结束了）。扫的过程中不断用 `far=max(far, i+nums[i])` 预估下一层能到多远。
3. **Step 3：** 当 `i==end`（当前层扫完了）就结算一次：跳数加一，把下一层边界 `end` 更新为 `far`。如果结算时发现 `far==end`，说明前沿原地踏步，永远到不了终点，返回 -1。
4. **Step 4：** 一旦新的 `end` 覆盖到最后一格，返回 `jumps`。

**手算演示一（“运行示例”部分 的前沿表，`a=[2,3,1,1,4]`）：**

| 位置 i | 跳长上界 x | 旧前沿 | 新前沿 far=max(旧, i+x) |
|---|---|---|---|
| 0 | 2 | 0 | 2 |
| 1 | 3 | 2 | 4 |
| 2 | 1 | 4 | 4 |
| 3 | 1 | 4 | 4 |
| 4 | 4 | 4 | 8 |

这张表就是 “运行示例”部分 运行后打印的内容。你看第 1 行到第 2 行：位置 1 的"1+3=4"把前沿从 2 推到 4，终点（下标 4）已被覆盖，所以 `can_jump` 在 i=1 那轮就返回 True。

**手算演示二（min_jumps 的分层过程，同一个数组 `[2,3,1,1,4]`）：**

- 初始：`end=0, far=0, jumps=0`。
- i=0：`far=max(0,0+2)=2`；`i==end`（0==0），当前层结算——`far(2) != end(0)` 说明前沿在长，`jumps=1`，`end=2`。含义：1 跳最远到位置 2。
- i=1：`far=max(2,1+3)=4`；`i(1) != end(2)`，还不结算。
- i=2：`far=max(4,2+1)=4`；`i==end`（2==2），结算——`jumps=2`，`end=4`。含义：2 跳最远到位置 4。`end>=4` 已覆盖最后一格，返回 2。

对照直觉：第 0 层是 {0}，第 1 层是位置 1~2，第 2 层是位置 3~4——每一层确实是一段连续区间，这就是我们只存右边界就够的原因。

## 4. 核心代码：can_jump

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def can_jump(nums):
    if not nums:
        return False
    far = 0
    for i, x in enumerate(nums):
        if i > far:
            return False
        far = max(far, i + x)
        if far >= len(nums) - 1:
            return True
    return False
```

### 逐段读懂代码

**函数一：能不能跳到（LC55 口径，以下是 “分段实现”部分 的逐字代码）**

```python
def can_jump(nums):
    if not nums:
        return False
    far = 0
    for i, x in enumerate(nums):
        if i > far:
            return False
        far = max(far, i + x)
        if far >= len(nums) - 1:
            return True
    return False
```

- `if not nums: return False`：空数组按本章契约直接判 False（连起点都没有，谈不上到达）。这是在挡空输入这个边界。
- `far=0`：0 跳的前沿就是起点自己。
- `if i>far: return False`：扫到位置 i 时如果前沿还没覆盖到 i，说明 i 是个空洞；又因为前沿是"连续区间的右边界"，i 后面的位置也全在洞里，可以立刻判负。
- `far=max(far,i+x)`：贪心领先性——只要 i 可达，它一步能摸到的最远处 `i+x` 就纳入前沿。
- `if far>=len(nums)-1: return True`：前沿够到最后一格就赢了，提前返回避免白扫。

**函数二：最少几跳（LC45 口径，不可达返回 -1）**

```python
def min_jumps(nums):
    if not nums:
        return -1
    end = far = jumps = 0
    for i in range(len(nums) - 1):
        if i > far:
            return -1
        far = max(far, i + nums[i])
        if i == end:
            if far == end:
                return -1
            jumps += 1
            end = far
            if end >= len(nums) - 1:
                return jumps
    return jumps
```

- `if not nums: return -1`：空输入的不可达契约（注意它和 `can_jump` 的空输入口径不同：一边 False、一边 -1，两者都表示"到不了"）。
- `end=far=jumps=0`：三个状态一起初始化——当前层右边界、下一层右边界、已用跳数。
- `for i in range(len(nums)-1)`：只扫到倒数第二格。为什么不到最后一格？因为跳数只在"离开一层"时增加，站在最后一格上不需要再跳，扫它没有意义；单元素数组（如 `[0]`）会直接跳过循环返回 `jumps=0`，正好符合"起点即终点，0 跳"。
- `if i>far: return -1`：和 can_jump 同款判负——出现不可达空洞。
- `far=max(far,i+nums[i])`：顺手为下一层囤积边界。
- `if i==end:`：位置 i 恰好是当前层右边界，意味着这一层扫完了，该结算跳数。
  - `if far==end: return -1`：前沿没有超过当前层边界，说明下一层是空的——不管再跳多少次都困在这一层里，判死。这就是 “思路推导”部分 说的"边界不再增长时，后面的位置永远不可达"。
  - `jumps+=1; end=far`：正式进入下一层。
  - `if end>=len(nums)-1: return jumps`：下一层已经把终点圈进来了，提前收工。

两个函数的状态对照（它们共享"前沿"思想，但 min_jumps 多带了一层簿记）：

| 对照项 | can_jump | min_jumps |
|---|---|---|
| 状态变量 | `far`（前沿） | `far` + `end` + `jumps` |
| 判负条件 | `i>far`（出现空洞） | `i>far` 或 `far==end`（前沿停滞） |
| 提前成功返回 | `far>=len(nums)-1` 返回 True | `end>=len(nums)-1` 返回 `jumps` |
| 空输入口径 | 返回 False | 返回 -1 |
| 时间 / 空间 | O(n) / O(1) | O(n) / O(1) |

## 5. 分段实现：按顺序运行

**本章接口：** `can_jump(nums)`、`min_jumps(nums)`

### can_jump

In [1]:
def can_jump(nums):
    if not nums:
        return False
    far = 0
    for i, x in enumerate(nums):
        if i > far:
            return False
        far = max(far, i + x)
        if far >= len(nums) - 1:
            return True
    return False

### min_jumps

In [2]:
def min_jumps(nums):
    if not nums:
        return -1
    end = far = jumps = 0
    for i in range(len(nums) - 1):
        if i > far:
            return -1
        far = max(far, i + nums[i])
        if i == end:
            if far == end:
                return -1
            jumps += 1
            end = far
            if end >= len(nums) - 1:
                return jumps
    return jumps

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [3]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

a=[2,3,1,1,4]; far=0; rows=[]
for i,x in enumerate(a):
    old=far; far=max(far,i+x); rows.append((i,x,old,far))
show_table(['位置','跳长上界','旧前沿','新前沿'],rows)

位置,跳长上界,旧前沿,新前沿
0,2,0,2
1,3,2,4
2,1,4,4
3,1,4,4
4,4,4,8


## 7. 正确性、适用条件与复杂度

层内所有顶点的出边并集决定下一层边界。扫描到当前层末端后才增加跳数，完全对应BFS层数，因此是最少跳数，而不是贪心地立即跳到最远位置。

**代价：** O(n)时间、O(1)辅助空间。跳长必须为非负整数。不可达扩展契约：min_jumps返回−1；空输入can_jump返回False。

### 展开理解

**为什么 can_jump 对？** `far` 的含义始终是"从位置 0 出发、用任意多跳，目前已确认能摸到的最远位置"。归纳着说：位置 i 能被 `far` 覆盖，就意味着 i 可达；i 可达则 i 能一步摸到 `i+x`，所以 `max(far, i+x)` 依然都是真可达的位置。反过来，扫描顺序是从左到右，第一个满足 `i>far` 的位置就是"前沿之外的第一个洞"，而所有跳跃都只能向右，洞后面的位置不可能绕过来，判 False 是安全的。

**为什么 min_jumps 是最少跳数？** 关键是把跳跃过程看成 BFS：位置是点、每个跳跃是一条边，"k 跳能到的位置集合"恰好是 BFS 的第 k 层。本章利用了一个性质——同一层的所有位置的"一步可达范围"并起来还是一段连续区间（因为每个位置 i 能到 i+1..i+nums[i]，连成一片），所以第 k+1 层就是"第 k 层所有位置一步可达的并集"，它的右边界正是扫描第 k 层时累计出的 `far`。代码里只有当扫描指针 i 走完当前层（`i==end`）才把 `jumps` 加一，这与 BFS"处理完整层再深入下一层"的计数节奏完全一致，所以 `jumps` 就是 BFS 层数，也就是最短跳数。“正确性与复杂度”部分 特别强调：这不是"每一步立刻跳到最远落点"那种贪心——那种逐落点贪心并不总是最优（见练习 1），本章做的是**层边界贪心**。

**复杂度。** 分开数更清楚：

- 两个函数都只把数组从左到右扫一遍，每个位置做常数次比较和赋值，所以时间是 O(n)、额外空间只有三个变量，是 O(1)。
- 拿具体数字感受：n=10⁵ 的数组就是十万次循环、二十万来次比较，一眨眼跑完。
- 对比朴素 BFS（把每个位置的可跳边都真正展开）：最坏要 O(n²) 条边，n=10⁵ 时是百亿级，根本跑不动——这就是"把可达集合压缩成一个右边界"省下来的量级。
- 前提：跳长必须是非负整数，代码里没有再校验，喂负数属于违反契约。

## 8. 单元测试：每个用例在检查什么

“自动测试”部分 的测试分两层。

**第一层：三条固定断言。**

- `assert can_jump([2,3,1,1,4]) and min_jumps([2,3,1,1,4])==2`：正常值测试，就是正文手算的例子，锁定"可达 + 最少 2 跳"这个标准答案。
- `assert not can_jump([3,2,1,0,4]) and min_jumps([3,2,1,0,4])==-1`：特殊值测试——中间的 0 是个"陷阱格"，任何路线都会被它吸住。手算：位置 0/1/2 的跳长再大，前沿最多推到 3；位置 3 跳长 0，前沿停在 3，`far==end` 触发，返回 -1。这条同时验证两个函数的不可达契约。
- `assert min_jumps([0])==0`：边界测试——单元素数组起点即终点，循环体一次都不执行，`jumps` 保持 0。它专门盯防"手滑把答案写成 1 或 -1"的实现。

**第二层：穷举对拍 BFS。** 对长度 n=1..6、每位取值 0/1/2 的**所有**数组（`product(range(3), repeat=n)` 是穷举不是抽样，n≤6 时最多 3⁶=729 个，全测得起），用一个真正的 BFS 当参照：`dist[i]` 记录从 0 到 i 的最少跳数（初始 -1 表示未达），从每个位置 i 把它能跳到的 j 逐个入队。然后两条断言：`min_jumps(a)==dist[-1]`（含"到不了时两边都是 -1"）和 `can_jump(a)==(dist[-1]!=-1)`（可达性等价于终点有距离）。因为测试覆盖了小规模的所有可能形态——全是 0、恰好差一步、贴边到达——它相当于在小世界里做了一次穷尽验证。

In [4]:
assert can_jump([2, 3, 1, 1, 4]) and min_jumps([2, 3, 1, 1, 4]) == 2

assert not can_jump([3, 2, 1, 0, 4]) and min_jumps([3, 2, 1, 0, 4]) == -1

assert min_jumps([0]) == 0

from itertools import product

from collections import deque

for n in range(1, 7):
    for a in product(range(3), repeat=n):
        dist = [-1] * n
        dist[0] = 0
        q = deque([0])
        while q:
            i = q.popleft()
            for j in range(i + 1, min(n, i + a[i] + 1)):
                if dist[j] == -1:
                    dist[j] = dist[i] + 1
                    q.append(j)
        assert min_jumps(a) == dist[-1]
        assert can_jump(a) == (dist[-1] != -1)

print('N094: 所有本章断言通过')

N094: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 解释为什么选择当前能跳最远的落点不等于正确的前沿算法。

**练习 2：** 加入不可达教学变体。

<details>
<summary>展开思路提示（不是完整答案）</summary>

**练习 1（为什么"每步跳到当前最远落点"不对）**：思路是找一个"最远落点跳长很短、次远落点跳长很长"的输入。提示方向：还是 `[2,3,1,1,4]`——站在位置 0 时最远能落位置 2（跳长只有 1），而落位置 1（跳长 3）反而一步登顶。逐落点贪心会走出 0→2→3→4 共 3 跳，而正确答案是 2 跳。你要写清楚：错误策略错在"把落点绑定死"，而本章算法只维护**层边界**，层的概念天然包含了"落在层内任何位置"的所有可能性，所以不漏最优路线。验证时可以手算两种策略的跳数对比，或者给逐落点贪心写个小实现拿随机数组对拍本章接口。

**练习 2（不可达变体）**：思路是把 -1 契约的触发路径逐条列出来。提示方向：至少构造三种输入——(a) “学习目标”部分就是 0 的 `[0,1]`（起点即死地）；(b) 前沿被 0 卡住的 `[3,2,1,0,4]`；(c) 前沿增长停滞的更长例子，比如 `[1,0,3]`（1 跳到位置 1，位置 1 跳长 0，`far==end`）。你要写明每种输入下手算的前沿演化过程，说明是代码里哪一行（`if i>far` 还是 `if far==end`）把它判死的，再用本章接口跑出 -1 与 False 对上号。边界别忘了空数组 `[]`：两个函数分别返回 -1 和 False。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [5]:
# 完整实现：本单元格不依赖其他单元格。
def can_jump(nums):
    if not nums:
        return False
    far = 0
    for i, x in enumerate(nums):
        if i > far:
            return False
        far = max(far, i + x)
        if far >= len(nums) - 1:
            return True
    return False

def min_jumps(nums):
    if not nums:
        return -1
    end = far = jumps = 0
    for i in range(len(nums) - 1):
        if i > far:
            return -1
        far = max(far, i + nums[i])
        if i == end:
            if far == end:
                return -1
            jumps += 1
            end = far
            if end >= len(nums) - 1:
                return jumps
    return jumps

# 示例与回归测试
assert can_jump([2, 3, 1, 1, 4]) and min_jumps([2, 3, 1, 1, 4]) == 2

assert not can_jump([3, 2, 1, 0, 4]) and min_jumps([3, 2, 1, 0, 4]) == -1

assert min_jumps([0]) == 0

from itertools import product

from collections import deque

for n in range(1, 7):
    for a in product(range(3), repeat=n):
        dist = [-1] * n
        dist[0] = 0
        q = deque([0])
        while q:
            i = q.popleft()
            for j in range(i + 1, min(n, i + a[i] + 1)):
                if dist[j] == -1:
                    dist[j] = dist[i] + 1
                    q.append(j)
        assert min_jumps(a) == dist[-1]
        assert can_jump(a) == (dist[-1] != -1)

print('N094: 所有本章断言通过')

N094: 所有本章断言通过


## 11. 代表题与迁移

- **55. Jump Game（canonical）**：这题就是 `can_jump` 的原题——只问可达性。它练的是本章"前沿压缩"思想的最简形态：一个 `far` 变量扫一遍出答案。
- **45. Jump Game II（canonical）**：这题就是 `min_jumps` 的原题——问最少跳数（原题保证可达，本章额外扩展了返回 -1 的教学契约）。它练的是"当前层/下一层"的层次结算节奏，本质是隐式图 BFS 的贪心压缩版。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。